# 07 - LLM Final Judgment (ContraDoc)

Re-ranks NLI-flagged chunk pairs with an LLM that sees both sentences and returns:
- **is_contradiction** (bool) — binary verdict
- **contradiction_type** — one of ContraDoc's 8 types (or `Not applicable`)
- **confidence** — low / medium / high
- **explanation** — one-sentence rationale

## Role in the pipeline

NLI @0.95 hit ~5% precision with 91.7% pair recall. That 95/100 false-positive rate is unusable as a final decision. Step 07 re-ranks the NLI-flagged candidates with an LLM to push precision up. Expected: the LLM filters out topically-related-but-not-contradictory pairs (NLI's biggest failure mode), at the cost of one LLM call per candidate.

**Input:** `data/processed/ContraDoc/nli_predictions.jsonl` (filtered by `NLI_THRESHOLD`).
**Output:** `data/processed/ContraDoc/llm_judgments.jsonl` — same schema as input, augmented with `llm_is_contradiction`, `llm_contradiction_type`, `llm_confidence`, `llm_explanation`.

Uses `settings.llm_model` via `langchain-openai` with structured output.

In [ ]:
import json
from collections import Counter
from pathlib import Path
from typing import Literal

from langchain_openai import ChatOpenAI
from pydantic import BaseModel, Field
from tqdm import tqdm

from config import settings

INPUT_PATH = Path("data/processed/ContraDoc/nli_predictions.jsonl")
OUTPUT_PATH = Path("data/processed/ContraDoc/llm_judgments.jsonl")
NLI_THRESHOLD = 0.5

## Load NLI predictions and filter by threshold

Only candidates with `nli_contradiction >= NLI_THRESHOLD` go to the LLM. Lowering the threshold = more LLM calls, higher recall.

In [8]:
all_candidates = [json.loads(line) for line in INPUT_PATH.open(encoding="utf-8")]
print(f"Loaded {len(all_candidates)} NLI predictions")

n_total_gold = sum(1 for c in all_candidates if c["is_gold_pair"])
print(f"  Total gold pairs in NLI set: {n_total_gold}")

candidates = [c for c in all_candidates if c["nli_contradiction"] >= NLI_THRESHOLD]
print(f"\nFiltered to {len(candidates)} pairs with nli_contradiction >= {NLI_THRESHOLD}")
print(f"  Gold pairs retained after filter: {sum(1 for c in candidates if c['is_gold_pair'])}")

Loaded 2451 NLI predictions
  Total gold pairs in NLI set: 23

Filtered to 581 pairs with nli_contradiction >= 0.5
  Gold pairs retained after filter: 22


## Structured-output schema (ContraDoc taxonomy)

In [9]:
class LLMJudgment(BaseModel):
    is_contradiction: bool = Field(
        description="True iff the two sentences describe the same entity, event, or relationship with incompatible attributes, OR cannot both be true in the same document context. Not contradictory: different entities, different time points in a narrative, one-sentence-more-specific-than-the-other."
    )
    contradiction_type: Literal[
        "Negation",
        "Numeric",
        "Content",
        "Perspective/View/Opinion",
        "Emotion/Mood/Feeling",
        "Relation",
        "Factual",
        "Causal",
        "Not applicable",
    ] = Field(
        description="Most applicable contradiction type from the ContraDoc taxonomy. Set to 'Not applicable' when is_contradiction is false."
    )
    confidence: Literal["low", "medium", "high"] = Field(
        description="Self-assessed confidence. 'low' for ambiguous cases, 'high' for clear-cut contradictions or clear-cut non-contradictions."
    )
    explanation: str = Field(
        description="One sentence rationale naming the specific attribute or claim that conflicts (or the reason the pair is NOT contradictory)."
    )

## LLM client and prompt

In [10]:
SYSTEM_PROMPT = """You are judging whether two sentences from the same document contradict each other.

A PAIR IS CONTRADICTORY when:
- Both sentences describe the same entity/event/relationship but assert incompatible attributes.
- They cannot both be true in the same document context (ignoring obvious narrative time-series where attributes legitimately change).

A PAIR IS NOT CONTRADICTORY when:
- The sentences talk about different entities, different works, or different time points that can coexist.
- One sentence is a specialization / paraphrase of the other without conflict.
- They share vocabulary but assert compatible facts.

If it IS a contradiction, classify into ONE type using the ContraDoc taxonomy:
- Negation: explicit negation flips polarity ('X did Y' vs 'X did NOT do Y').
- Numeric: number, date, or quantity mismatch about the same entity.
- Content: a non-numeric, non-subjective attribute of the same entity/event is swapped (kidney 'to a stranger' vs 'to her friend').
- Perspective/View/Opinion: conflicting subjective evaluations ('praised' vs 'disliked').
- Emotion/Mood/Feeling: conflicting inner states ('worriedly' vs 'happily').
- Relation: mutually exclusive relations between entities ('married couple' vs 'sister').
- Factual: real-world factual mismatch (location, identity, profession, etc.).
- Causal: the effect does not match the stated cause.

Return structured output: is_contradiction (bool), contradiction_type (one of the above or 'Not applicable'), confidence (low/medium/high), explanation (one sentence).
"""

llm = ChatOpenAI(model=settings.llm_model, api_key=settings.openai_api_key, temperature=0)
judge = llm.with_structured_output(LLMJudgment)


def judge_pair(sentence_a: str, sentence_b: str) -> LLMJudgment:
    user_msg = f"Sentence A: {sentence_a}\n\nSentence B: {sentence_b}\n\nReturn your judgment."
    return judge.invoke(
        [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_msg},
        ]
    )

## Sanity check on 3 pairs

Quick spot-check before the full run. One clear gold, one likely FP from NLI, and one neutral pair.

In [11]:
tests = [
    (
        "Capel Lligwy is a well-preserved chapel near Rhos Lligwy in Anglesey.",
        "Capel Lligwy is a ruined chapel near Rhos Lligwy in Anglesey.",
    ),
    (
        "In 2006 Boulter starred in the play Citizenship written by Mark Ravenhill.",
        "Boulter starred in the 2011 film Mercenaries directed by Paris Leonti.",
    ),
    (
        "The cat is sleeping on the mat.",
        "The dog is barking loudly.",
    ),
]
for a, b in tests:
    j = judge_pair(a, b)
    print(f"A: {a}")
    print(f"B: {b}")
    print(f"  -> contradiction={j.is_contradiction}  type={j.contradiction_type}  conf={j.confidence}")
    print(f"  -> {j.explanation}")
    print()

A: Capel Lligwy is a well-preserved chapel near Rhos Lligwy in Anglesey.
B: Capel Lligwy is a ruined chapel near Rhos Lligwy in Anglesey.
  -> contradiction=True  type=Content  conf=high
  -> The two sentences make incompatible claims about the chapel’s condition, describing Capel Lligwy as both well-preserved and ruined.

A: In 2006 Boulter starred in the play Citizenship written by Mark Ravenhill.
B: Boulter starred in the 2011 film Mercenaries directed by Paris Leonti.
  -> contradiction=False  type=Not applicable  conf=high
  -> The sentences describe Boulter appearing in two different works in different years, which can both be true.

A: The cat is sleeping on the mat.
B: The dog is barking loudly.
  -> contradiction=False  type=Not applicable  conf=high
  -> The sentences describe different entities and actions, so they can both be true at the same time.



## Run the LLM on every filtered candidate (resumable)

In [12]:
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)


def cand_key(c):
    return (c["doc_id"], c["chunk_a"]["sentence_id"], c["chunk_b"]["sentence_id"])


done_keys: set[tuple] = set()
if OUTPUT_PATH.exists():
    with OUTPUT_PATH.open(encoding="utf-8") as f:
        for line in f:
            done_keys.add(cand_key(json.loads(line)))
    print(f"Resuming: {len(done_keys)} pairs already judged")

todo = [c for c in candidates if cand_key(c) not in done_keys]
print(f"To judge: {len(todo)} pairs")

with OUTPUT_PATH.open("a", encoding="utf-8") as f:
    for c in tqdm(todo):
        try:
            j = judge_pair(c["chunk_a"]["source_text"], c["chunk_b"]["source_text"])
        except Exception as exc:
            print(f"FAILED {cand_key(c)}: {type(exc).__name__}: {exc}")
            continue
        rec = dict(c)
        rec["llm_is_contradiction"] = j.is_contradiction
        rec["llm_contradiction_type"] = j.contradiction_type
        rec["llm_confidence"] = j.confidence
        rec["llm_explanation"] = j.explanation
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")
        f.flush()

print(f"Done. Output: {OUTPUT_PATH.resolve()}")

To judge: 581 pairs


  0%|          | 0/581 [00:00<?, ?it/s]

Done. Output: D:\AT82.05-Claim-Contradiction-Over-Knowledge-Graphs\experiments\data\processed\ContraDoc\llm_judgments.jsonl


## Evaluation: LLM vs NLI, plus end-to-end numbers

Two views:
1. **On the NLI-filtered subset** — how well does the LLM re-rank the NLI positives?
2. **End-to-end vs the full gold set** — combining retrieval + NLI + LLM.

In [13]:
records = [json.loads(line) for line in OUTPUT_PATH.open(encoding="utf-8")]
print(f"Loaded {len(records)} LLM judgments")

tp = sum(1 for r in records if r["llm_is_contradiction"] and r["is_gold_pair"])
fp = sum(1 for r in records if r["llm_is_contradiction"] and not r["is_gold_pair"])
fn_filtered = sum(1 for r in records if not r["llm_is_contradiction"] and r["is_gold_pair"])
tn = sum(1 for r in records if not r["llm_is_contradiction"] and not r["is_gold_pair"])

prec = tp / max(tp + fp, 1)
rec_filt = tp / max(tp + fn_filtered, 1)
f1 = 2 * prec * rec_filt / max(prec + rec_filt, 1e-9)

print(f"\n=== LLM on the NLI-filtered subset ({len(records)} pairs, {tp + fn_filtered} gold) ===")
print(f"  TP={tp}  FP={fp}  FN={fn_filtered}  TN={tn}")
print(f"  Precision: {prec:.1%}")
print(f"  Recall (vs filtered set): {rec_filt:.1%}")
print(f"  F1: {f1:.1%}")

end_to_end_r = tp / max(n_total_gold, 1)
print(f"\n=== End-to-end: retrieval + NLI + LLM vs {n_total_gold} gold pairs ===")
print(f"  TP caught: {tp}")
print(f"  End-to-end pair recall: {end_to_end_r:.1%}")
print(f"  End-to-end precision:   {prec:.1%}")
print(f"  End-to-end F1:          {2 * prec * end_to_end_r / max(prec + end_to_end_r, 1e-9):.1%}")

doc_ids_with_tp = {r["doc_id"] for r in records if r["llm_is_contradiction"] and r["is_gold_pair"]}
n_gold_docs_total = len({c["doc_id"] for c in all_candidates if c["is_gold_pair"]})
print(f"  End-to-end doc recall: {len(doc_ids_with_tp)}/{n_gold_docs_total} = {len(doc_ids_with_tp) / max(n_gold_docs_total, 1):.1%}")

type_dist = Counter(r["llm_contradiction_type"] for r in records if r["llm_is_contradiction"])
print(f"\n=== Predicted contradiction types (LLM positives, n={sum(type_dist.values())}) ===")
for t, n in type_dist.most_common():
    print(f"  {t}: {n}")

conf_dist = Counter(r["llm_confidence"] for r in records)
print(f"\n=== Confidence distribution (all judgments) ===")
for c, n in conf_dist.most_common():
    print(f"  {c}: {n}")

Loaded 581 LLM judgments

=== LLM on the NLI-filtered subset (581 pairs, 22 gold) ===
  TP=18  FP=131  FN=4  TN=428
  Precision: 12.1%
  Recall (vs filtered set): 81.8%
  F1: 21.1%

=== End-to-end: retrieval + NLI + LLM vs 23 gold pairs ===
  TP caught: 18
  End-to-end pair recall: 78.3%
  End-to-end precision:   12.1%
  End-to-end F1:          20.9%
  End-to-end doc recall: 18/23 = 78.3%

=== Predicted contradiction types (LLM positives, n=149) ===
  Numeric: 48
  Content: 46
  Factual: 37
  Negation: 11
  Relation: 3
  Perspective/View/Opinion: 2
  Causal: 1
  Emotion/Mood/Feeling: 1

=== Confidence distribution (all judgments) ===
  high: 539
  medium: 41
  low: 1


## Sample true-positives and false-positives (post-LLM)

In [14]:
tp_records = [r for r in records if r["llm_is_contradiction"] and r["is_gold_pair"]][:5]
fp_records = [r for r in records if r["llm_is_contradiction"] and not r["is_gold_pair"]][:5]
fn_records = [r for r in records if not r["llm_is_contradiction"] and r["is_gold_pair"]][:5]

print("=== Top 5 TRUE positives (LLM flagged as contradiction AND gold) ===")
for r in tp_records:
    print(f"  doc={r['doc_id']}  type={r['llm_contradiction_type']}  conf={r['llm_confidence']}")
    print(f"    A: {r['chunk_a']['source_text'][:140]}")
    print(f"    B: {r['chunk_b']['source_text'][:140]}")
    print(f"    reason: {r['llm_explanation']}")
    print()

print("=== Top 5 FALSE positives (LLM flagged but not gold — likely real but unannotated, or genuine FP) ===")
for r in fp_records:
    print(f"  doc={r['doc_id']}  type={r['llm_contradiction_type']}  conf={r['llm_confidence']}")
    print(f"    A: {r['chunk_a']['source_text'][:140]}")
    print(f"    B: {r['chunk_b']['source_text'][:140]}")
    print(f"    reason: {r['llm_explanation']}")
    print()

print("=== Top 5 FALSE negatives (LLM rejected but gold says contradiction) ===")
for r in fn_records:
    print(f"  doc={r['doc_id']}  type={r['llm_contradiction_type']}  conf={r['llm_confidence']}")
    print(f"    A: {r['chunk_a']['source_text'][:140]}")
    print(f"    B: {r['chunk_b']['source_text'][:140]}")
    print(f"    reason: {r['llm_explanation']}")
    print()

=== Top 5 TRUE positives (LLM flagged as contradiction AND gold) ===
  doc=3499318675_7  type=Negation  conf=high
    A: What became NY 31B was not originally designated as NY 293 as part of the 1930 renumbering of state highways in New York.
    B: What became NY 31B was originally designated as NY 293 as part of the 1930 renumbering of state highways in New York.
    reason: Sentence A explicitly denies that the route was originally designated as NY 293, while Sentence B explicitly affirms that it was.

  doc=3489738250_7  type=Content  conf=high
    A: Trey is looking to take up engineering as a college major.
    B: Trey is looking to take up special education as a college major, in addition to playing basketball in the fall.
    reason: The two sentences assign Trey different intended college majors—engineering versus special education—which are incompatible as stated.

  doc=3488771853_5  type=Content  conf=high
    A: He describes the inevitability of death for all things at the

## Per-type recall breakdown (LLM judgments)

Per ContraDoc contradiction type — what does the LLM catch vs. miss? Multi-label docs contribute to every listed type.

In [15]:
from collections import defaultdict

doc_types = {}
with Path("data/processed/ContraDoc/triples.jsonl").open(encoding="utf-8") as f:
    for line in f:
        r = json.loads(line)
        if r["contradiction"] == "YES":
            doc_types[r["doc_id"]] = [t for t in (r.get("contra_type") or "none").split("|") if t]


def rec_key(r):
    return tuple(sorted([(r["doc_id"], r["chunk_a"]["sentence_id"]), (r["doc_id"], r["chunk_b"]["sentence_id"])]))


gold_set = {rec_key(r) for r in records if r["is_gold_pair"]}
llm_positive = {rec_key(r) for r in records if r["llm_is_contradiction"]}

# Also compare: NLI-filtered-input set (pre-LLM) vs post-LLM
nli_filtered = {rec_key(r) for r in records}  # everything at this stage already passed NLI threshold


def per_type_recall(pairs, name):
    type_totals = defaultdict(int)
    type_caught = defaultdict(int)
    for p in gold_set:
        doc_id = p[0][0]
        for t in doc_types.get(doc_id, ["unknown"]):
            type_totals[t] += 1
            if p in pairs:
                type_caught[t] += 1
    all_types = sorted(type_totals.keys(), key=lambda x: -type_totals[x])
    print(f"\n{name}:")
    print(f"  {'type':30s}  caught  total  recall")
    print("  " + "-" * 52)
    for t in all_types:
        caught, total = type_caught[t], type_totals[t]
        print(f"  {t:30s}  {caught:>6}  {total:>5}  {caught / max(total, 1):>6.1%}")


per_type_recall(nli_filtered, "NLI-filtered input (pre-LLM)")
per_type_recall(llm_positive, "LLM positive (post-LLM)")

# Confusion matrix of predicted type vs true type (for LLM-positive gold pairs)
print("\n=== LLM predicted type vs true doc type (for gold pairs that LLM flagged) ===")
from collections import Counter

confusion = defaultdict(lambda: Counter())
for r in records:
    if not (r["llm_is_contradiction"] and r["is_gold_pair"]):
        continue
    doc_id = r["doc_id"]
    for true_t in doc_types.get(doc_id, ["unknown"]):
        confusion[true_t][r["llm_contradiction_type"]] += 1
for true_t, counts in sorted(confusion.items(), key=lambda x: -sum(x[1].values())):
    print(f"  true={true_t}: {dict(counts)}")


NLI-filtered input (pre-LLM):
  type                            caught  total  recall
  ----------------------------------------------------
  Content                             15     15  100.0%
  Numeric                              7      7  100.0%
  Perspective/View/Opinion             4      4  100.0%
  Negation                             3      3  100.0%
  Factual                              2      2  100.0%
  Emotion/Mood/Feeling                 2      2  100.0%

LLM positive (post-LLM):
  type                            caught  total  recall
  ----------------------------------------------------
  Content                             13     15   86.7%
  Numeric                              6      7   85.7%
  Perspective/View/Opinion             4      4  100.0%
  Negation                             3      3  100.0%
  Factual                              2      2  100.0%
  Emotion/Mood/Feeling                 0      2    0.0%

=== LLM predicted type vs true doc type (for gol